In [1]:
# CELL 0: Imports & Configuration
import re
import json
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd
from unidecode import unidecode

print('✓ Imports successful')

# ==== FILE PATHS ====
PATH_KEYWORDS = Path('keywords_fixed.json')
PATH_INPUT = Path('CRM raw.xlsx')
PATH_OUTPUT = Path('CRM_classified.xlsx')
PATH_OUTPUT_LLM = Path('CRM_classified_with_LLM.xlsx')

# ==== SOURCE COLUMNS (5 text columns) ====
COL_R = 'Tình trạng hiện tại'
COL_S = 'Tình hình tiến độ công trình'
COL_T = 'Nội dung làm việc, yêu cầu KH & đánh giá'
COL_U = 'Kế hoạch lần tới'
COL_V = 'Đề xuất'

# ==== OUTPUT (15 MINOR columns, same as old pipeline) ====
MINOR_COLUMNS_ORDER: List[Tuple[str, str]] = [
    ('Hoạt Động CRM', 'Tình hình hiện tại'),
    ('Hoạt Động CRM', 'Tiến độ'),
    ('Hoạt Động CRM', 'ngày lấy hàng'),
    ('Đối Thủ Cạnh Tranh', 'Các Hãng đối thủ cạnh tranh'),
    ('Đối Thủ Cạnh Tranh', 'Đối tượng'),
    ('Đối Thủ Cạnh Tranh', 'Nội dung làm việc'),
    ('Đối Thủ Cạnh Tranh', 'Lợi thế'),
    ('AETT', 'Đối tượng'),
    ('AETT', 'Nội dung làm việc'),
    ('AETT', 'Nhận xét tiếp thị'),
    ('Khách Hàng', 'Ý kiến KH'),
    ('Khách Hàng', 'Nhận xét KH'),
    ('Kế Hoạch', 'Ngày làm việc/ giao hàng:'),
    ('Kế Hoạch', 'Kế hoạch lần tới'),
    ('Kế Hoạch', 'Đề xuất'),
]

def col_name(major: str, minor: str) -> str:
    return f'[{major}] {minor}'

OUTPUT_COLUMNS = [col_name(major, minor) for major, minor in MINOR_COLUMNS_ORDER]

print(f'✓ Config loaded: {len(OUTPUT_COLUMNS)} output columns')

✓ Imports successful
✓ Config loaded: 15 output columns


In [2]:
# CELL 0.5: API Key + Model (Gemini)
# NOTE: Keep this key private. Avoid committing to git.
import os

API_KEY = "YOUR_GEMINI_API_KEY_HERE"
os.environ['GEMINI_API_KEY'] = API_KEY
os.environ['GEMINI_MODEL'] = 'models/gemini-2.0-flash'
print('✓ API key and model configured')


✓ API key and model configured


In [3]:
# CELL 1: Utility Functions (match old pipeline)

def norm_text(s: Any, keep_vn: bool = True) -> str:
    s = '' if s is None else str(s)
    s = s.strip().lower()
    s = re.sub(r'\s+', ' ', s)
    if not keep_vn:
        s = unidecode(s)
    return s

def split_terms(value: Any) -> List[str]:
    if value is None:
        return []
    text = str(value).replace('\r', '\n')
    parts: List[str] = []
    for line in text.split('\n'):
        line = line.strip()
        if not line:
            continue
        for seg in [p.strip() for p in line.split(';') if p.strip()]:
            if ',' in seg and not re.search(r'\d,\d', seg):
                for seg2 in [x.strip() for x in seg.split(',') if x.strip()]:
                    parts.append(seg2)
            else:
                parts.append(seg)
    seen, out = set(), []
    for p in parts:
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out

def compile_regex_set(terms: List[str]) -> List[re.Pattern]:
    regs: List[re.Pattern] = []
    for t in terms:
        t_clean = t.strip()
        if not t_clean:
            continue
        pattern = re.escape(t_clean)
        regs.append(re.compile(rf'(?<!\w){pattern}(?!\w)', re.IGNORECASE))
        if unidecode(t_clean) != t_clean:
            pattern2 = re.escape(unidecode(t_clean))
            regs.append(re.compile(rf'(?<!\w){pattern2}(?!\w)', re.IGNORECASE))
    return regs

def get_text_by_priority(row: Any, col_list: List[str]) -> str:
    for col in col_list:
        text = str(row.get(col, '')) if pd.notna(row.get(col)) else ''
        if text.strip() and text.strip().lower() not in ['nan', 'none', '']:
            return text.strip()
    return ''

def find_first_match(text: str, patterns: Dict[Tuple[str, str, str], List[re.Pattern]], major: str, minor: str) -> Optional[str]:
    for (m, f, sub), pats in patterns.items():
        if m == major and f == minor:
            if any(p.search(text) or p.search(unidecode(text)) for p in pats):
                return sub
    return None

def find_deepest_match(text: str, patterns: Dict[Tuple[str, str, str], List[re.Pattern]], major: str, minor: str) -> Optional[str]:
    matches: List[str] = []
    for (m, f, sub), pats in patterns.items():
        if m == major and f == minor:
            if any(p.search(text) or p.search(unidecode(text)) for p in pats):
                matches.append(sub)
    if not matches:
        return None
    priority_map = {
        'Tư vấn/ khảo sát': 1,
        'Báo giá': 2,
        'Chốt giá': 3,
        'Cấp hàng': 4,
        'Lắp đặt/ cài đặt': 5,
        'Bảo hành': 6,
        'Chăm sóc ngắn hạn': 7,
        'Chăm sóc tiếp': 8,
        'Hỗ trợ kỹ thuật': 9,
        'Giải đáp thắc mắc': 10,
    }
    return max(matches, key=lambda x: priority_map.get(x, 0))

def has_price_indicator(text: str) -> bool:
    price_keywords = ['triệu', 'tỷ', 'vnđ', 'vnd', 'usd', 'đô', 'đồng']
    text_lower = (text or '').lower()
    return any(kw in text_lower for kw in price_keywords)

# ==== Date extraction (match old pipeline intent) ====
DATE_PATTERNS = [
    r'\b(\d{1,2}/\d{1,2}/\d{4})\b',
    r'\b(\d{1,2}/\d{1,2})\b',
    r'\b(\d{4}-\d{1,2}-\d{1,2})\b',
]
ETA_HINTS = ['giao hàng', 'cấp hàng', 'lấy hàng', 'eta', 'etd', 'dự kiến giao', 'dự kiến cấp']

def find_eta(texts: List[str]) -> Optional[str]:
    raw = ' '.join([t for t in texts if t])
    if not raw.strip():
        return None
    raw_low = norm_text(raw)
    if not any(h in raw_low for h in ETA_HINTS):
        return None
    for pat in DATE_PATTERNS:
        m = re.search(pat, raw, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    return None

def find_plan_date(texts: List[str]) -> Optional[str]:
    raw = ' '.join([t for t in texts if t])
    if not raw.strip():
        return None
    for pat in DATE_PATTERNS:
        m = re.search(pat, raw, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    return None

print('✓ Utility functions defined')

✓ Utility functions defined


In [4]:
# CELL 2: Load Keywords & Build Pattern Map (match old pipeline)

with open(PATH_KEYWORDS, 'r', encoding='utf-8') as f:
    KW = json.load(f)

print('✓ Loaded keywords_fixed.json')
print(f'  - {len(KW)} MAJOR groups')

def build_pattern_map() -> Dict[Tuple[str, str, str], List[re.Pattern]]:
    pattern_map: Dict[Tuple[str, str, str], List[re.Pattern]] = {}
    for major, fields_dict in KW.items():
        if not isinstance(fields_dict, dict):
            continue
        for minor, subs_dict in fields_dict.items():
            # Handle nested: "Cạnh Tranh" -> "Các Hãng đối thủ cạnh tranh" (brand list)
            if minor == 'Cạnh Tranh' and isinstance(subs_dict, dict):
                for nested_field, nested_content in subs_dict.items():
                    if isinstance(nested_content, list):
                        term_list: List[str] = []
                        for t in nested_content:
                            term_list.extend(split_terms(t))
                        if term_list:
                            pats = compile_regex_set(term_list)
                            # Map to minor = nested_field for classifier convenience
                            pattern_map[(major, nested_field, 'Brand match')] = pats
                continue
            # Standard: minor -> {sub: [keywords]}
            if isinstance(subs_dict, dict):
                for sub, keywords in subs_dict.items():
                    term_list: List[str] = []
                    if isinstance(keywords, list):
                        for t in keywords:
                            term_list.extend(split_terms(t))
                    elif isinstance(keywords, str):
                        term_list.extend(split_terms(keywords))
                    if term_list:
                        pattern_map[(major, minor, sub)] = compile_regex_set(term_list)
    return pattern_map

PATTERN_MAP = build_pattern_map()
print(f'✓ Built pattern map: {len(PATTERN_MAP)} patterns')

✓ Loaded keywords_fixed.json
  - 5 MAJOR groups
✓ Built pattern map: 94 patterns


In [5]:
# CELL 3: Read Input Data

df = pd.read_excel(PATH_INPUT)
print(f'✓ Loaded {len(df)} rows from {PATH_INPUT.name}')
print(f'  Columns sample: {list(df.columns)[:15]}')

# Normalize lightly the 5 source columns
for col in [COL_R, COL_S, COL_T, COL_U, COL_V]:
    if col in df.columns:
        df[col] = df[col].astype(str).fillna('').str.replace(r'\s+', ' ', regex=True).str.strip()
    else:
        df[col] = ''

# Sample row
sample_idx = 0
sample_row = df.iloc[sample_idx]
print(f'\n📝 Sample Row {sample_idx}:')
print(f'  R: {sample_row.get(COL_R, "")}')
print(f'  S: {sample_row.get(COL_S, "")}')
print(f'  T: {sample_row.get(COL_T, "")}')
print(f'  U: {sample_row.get(COL_U, "")}')
print(f'  V: {sample_row.get(COL_V, "")}')

d:\Works\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell P2382 is marked as a date but the serial value 220000000 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Works\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell P2390 is marked as a date but the serial value 20000000 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Works\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell P2391 is marked as a date but the serial value 20000000 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Works\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell P2392 is marked as a date but the serial value 46000000 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Works\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell P2403 is marked as

✓ Loaded 13803 rows from CRM raw.xlsx
  Columns sample: ['Ngày bắt đầu', 'Tên NV', 'Tên NV hỗ trợ', 'Mã CT', 'Tên CT', 'Loại đối tác', 'Tên đối tác', 'Địa chỉ', 'Người liên hệ', 'Điện thoại', 'Tình hình/kế hoạch', 'Thời gian ghi nhận tình hình', 'Tiến độ CT', 'Ngày kết thúc', 'Kết quả']

📝 Sample Row 0:
  R: nan
  S: nan
  T: Chốt lấy hàng
  U: nan
  V: nan


In [6]:
# CELL 4: Classify Single Row (RULES: source columns + multi-tag => 'mơ hồ')

def _clean_text(v: Any) -> str:
    if v is None:
        return ''
    s = str(v).strip()
    if not s or s.lower() in ['nan', 'none', 'null']:
        return ''
    return s

def _find_sub_matches(text: str, patterns: Dict[Tuple[str, str, str], List[re.Pattern]], major: str, minor: str) -> List[str]:
    if not text:
        return []
    matched: List[str] = []
    txt2 = unidecode(text)
    for (m, f, sub), pats in patterns.items():
        if m == major and f == minor:
            if any(p.search(text) or p.search(txt2) for p in pats):
                matched.append(sub)
    # unique, stable
    out: List[str] = []
    seen = set()
    for x in matched:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

def _pick_or_ambiguous(subs: List[str]) -> Optional[str]:
    if not subs:
        return None
    if len(subs) == 1:
        return subs[0]
    return 'mơ hồ'

def _find_competitor_brands(text: str, patterns: Dict[Tuple[str, str, str], List[re.Pattern]]) -> List[str]:
    """Return a list of matched competitor brands (multi allowed)."""
    if not text:
        return []
    txt2 = unidecode(text)
    brands: List[str] = []
    for (m, f, sub), pats in patterns.items():
        if m == 'Đối Thủ Cạnh Tranh' and f == 'Các Hãng đối thủ cạnh tranh':
            for p in pats:
                m1 = p.search(text) or p.search(txt2)
                if m1:
                    # keep original surface form but normalize title
                    brands.append(m1.group().strip().title())
    # unique, stable
    out: List[str] = []
    seen = set()
    for b in brands:
        k = b.lower()
        if k not in seen:
            out.append(b)
            seen.add(k)
    return out

def classify_row(row: Any, patterns: Dict[Tuple[str, str, str], List[re.Pattern]]) -> Dict[Tuple[str, str], str]:
    """Return {(major, minor): value} for a row.

    Rules implemented (per user spec):
    - 'Tình hình hiện tại' copies directly from column R.
    - Groups Hoạt Động CRM (except that R-copy col), Đối Thủ Cạnh Tranh, AETT, Khách Hàng use ONLY S+T text.
    - Group Kế Hoạch uses ONLY U+V text.
    - If keywords match >=2 tags (SUBs) for the same output column => set value = 'mơ hồ'.
    - AETT 'Nội dung làm việc' must have a value when T has text: if no keyword match => 'mơ hồ'.
    - AETT 'mơ hồ' also applies when S/T has real data but no keyword match.
    """
    result: Dict[Tuple[str, str], str] = {}

    text_r = _clean_text(row.get(COL_R, ''))
    text_s = _clean_text(row.get(COL_S, ''))
    text_t = _clean_text(row.get(COL_T, ''))
    text_u = _clean_text(row.get(COL_U, ''))
    text_v = _clean_text(row.get(COL_V, ''))

    text_st = ' | '.join([x for x in [text_s, text_t] if x])
    has_st_data = bool(text_s or text_t)

    # === Hoạt Động CRM ===
    # Copy trực tiếp từ R
    if text_r:
        result[('Hoạt Động CRM', 'Tình hình hiện tại')] = text_r

    # Tiến độ: multi-tag => mơ hồ
    subs_progress = _find_sub_matches(text_st, patterns, 'Hoạt Động CRM', 'Tiến độ')
    v_progress = _pick_or_ambiguous(subs_progress)
    if v_progress:
        result[('Hoạt Động CRM', 'Tiến độ')] = v_progress

    # ngày lấy hàng: để LLM xử lý chính xác (phân biệt cấp vs giao)

    # === Đối Thủ Cạnh Tranh ===
    brands = _find_competitor_brands(text_st, patterns)
    if brands:
        result[('Đối Thủ Cạnh Tranh', 'Các Hãng đối thủ cạnh tranh')] = '; '.join(brands)

    for field in ['Đối tượng', 'Nội dung làm việc', 'Lợi thế']:
        subs = _find_sub_matches(text_st, patterns, 'Đối Thủ Cạnh Tranh', field)
        v = _pick_or_ambiguous(subs)
        if v:
            result[('Đối Thủ Cạnh Tranh', field)] = v

    # === AETT ===
    subs_obj = _find_sub_matches(text_st, patterns, 'AETT', 'Đối tượng')
    v_obj = _pick_or_ambiguous(subs_obj)
    if v_obj:
        result[('AETT', 'Đối tượng')] = v_obj

    subs_work = _find_sub_matches(text_st, patterns, 'AETT', 'Nội dung làm việc')
    v_work = _pick_or_ambiguous(subs_work)

    # Price indicator: if no match but mentions money => minimally 'Báo giá'
    if v_work is None and has_price_indicator(text_st):
        v_work = 'Báo giá'

    # Enforce 'AETT work must have value' when T has text
    if v_work is None and text_t:
        v_work = 'mơ hồ'

    # Also label 'mơ hồ' when S/T has data but no keyword match
    if v_work is None and has_st_data:
        v_work = 'mơ hồ'

    if v_work:
        result[('AETT', 'Nội dung làm việc')] = v_work

    subs_mkt = _find_sub_matches(text_st, patterns, 'AETT', 'Nhận xét tiếp thị')
    v_mkt = _pick_or_ambiguous(subs_mkt)
    if v_mkt:
        result[('AETT', 'Nhận xét tiếp thị')] = v_mkt

    # === Khách Hàng ===
    for field in ['Ý kiến KH', 'Nhận xét KH']:
        subs = _find_sub_matches(text_st, patterns, 'Khách Hàng', field)
        v = _pick_or_ambiguous(subs)
        if v:
            result[('Khách Hàng', field)] = v

    # === Kế Hoạch (from U/V only) ===
    text_uv = ' | '.join([x for x in [text_u, text_v] if x])

    subs_plan = _find_sub_matches(text_uv, patterns, 'Kế Hoạch', 'Kế hoạch lần tới')
    v_plan = _pick_or_ambiguous(subs_plan)
    if v_plan:
        result[('Kế Hoạch', 'Kế hoạch lần tới')] = v_plan

    subs_prop = _find_sub_matches(text_v, patterns, 'Kế Hoạch', 'Đề xuất')
    v_prop = _pick_or_ambiguous(subs_prop)
    if v_prop:
        result[('Kế Hoạch', 'Đề xuất')] = v_prop

    # Date column left for LLM

    return result

print('✓ classify_row() updated (multi-tag => mơ hồ, source columns enforced)')


✓ classify_row() updated (multi-tag => mơ hồ, source columns enforced)


In [7]:
# CELL 5: Batch Process All Rows

print('Starting batch classification...')
all_classifications: List[Dict[Tuple[str, str], str]] = []
fuzzy_count = 0

for idx, row in df.iterrows():
    if idx % 2000 == 0:
        print(f'  {idx}/{len(df)}')
    cls = classify_row(row, PATTERN_MAP)
    all_classifications.append(cls)
    if cls.get(('AETT', 'Nội dung làm việc')) == 'mơ hồ':
        fuzzy_count += 1

print(f'✓ Classified {len(all_classifications)} rows')
print(f"⚠️  {fuzzy_count} rows labeled 'mơ hồ' (need LLM)")

Starting batch classification...
  0/13803
  2000/13803


KeyboardInterrupt: 

In [ ]:
# CELL 6: Export Results to Excel (base + 15 classified columns)

rows_out: List[Dict[str, Any]] = []
for idx, cls in enumerate(all_classifications):
    row_out: Dict[str, Any] = {'row_idx': idx}
    for major, minor in MINOR_COLUMNS_ORDER:
        row_out[col_name(major, minor)] = cls.get((major, minor), '')
    rows_out.append(row_out)

df_cls = pd.DataFrame(rows_out)
df_final = pd.concat([df.reset_index(drop=True), df_cls], axis=1)

df_final.to_excel(PATH_OUTPUT, index=False)
print(f'✓ Exported to {PATH_OUTPUT.name}')
print(f'  Shape: {df_final.shape}')

✓ Exported to CRM_classified.xlsx
  Shape: (13803, 40)


In [ ]:
# CELL 7: Identify Rows/Cells to Fill with LLM (blank or 'mơ hồ')

# We will ask LLM to fill only columns that are blank or 'mơ hồ'
COL_CURRENT_STATUS = col_name('Hoạt Động CRM', 'Tình hình hiện tại')

# LLM should not touch the copied column (it comes directly from R)
LLM_TARGET_COLS = [c for c in OUTPUT_COLUMNS if c != COL_CURRENT_STATUS]

COL_CRM_DATE_PICKUP = col_name('Hoạt Động CRM', 'ngày lấy hàng')
COL_PLAN_DATE = col_name('Kế Hoạch', 'Ngày làm việc/ giao hàng:')

DATE_HINT_RE = re.compile(
    r"(\b\d{1,2}[/-]\d{1,2}([/-]\d{2,4})?\b|\btháng\s*\d{1,2}\b|\bquý\s*[1-4]\b|\bđầu\s*tháng\b|\bcuối\s*tháng\b)",
    flags=re.IGNORECASE,
 )

def _is_blank_or_fuzzy(v: Any) -> bool:
    if v is None:
        return True
    s = str(v).strip()
    if not s:
        return True
    return s.lower() == 'mơ hồ'

def _norm_text(v: Any) -> str:
    if v is None:
        return ''
    s = str(v).strip()
    if s.lower() == 'nan':
        return ''
    return s

def _has_text(*vals: Any) -> bool:
    return any(bool(_norm_text(v)) for v in vals)

def _has_date_hint(text: str) -> bool:
    return bool(text and DATE_HINT_RE.search(text))

missing_cols_per_row: List[List[str]] = []
need_llm_mask: List[bool] = []

for _, row in df_final.iterrows():
    s_txt = _norm_text(row.get(COL_S))
    t_txt = _norm_text(row.get(COL_T))
    u_txt = _norm_text(row.get(COL_U))
    v_txt = _norm_text(row.get(COL_V))

    st = (s_txt + "\n" + t_txt).strip()
    uv = (u_txt + "\n" + v_txt).strip()

    missing: List[str] = []
    for c in LLM_TARGET_COLS:
        if not _is_blank_or_fuzzy(row.get(c)):
            continue

        # Column-source constraints:
        # - Hoạt Động CRM / Đối Thủ / AETT / Khách Hàng use S,T
        # - Kế Hoạch uses U,V
        # Only ask LLM when there is relevant source text (or date hints for date columns).
        if c.startswith('[Kế Hoạch]'):
            if not uv:
                continue
            if c == COL_PLAN_DATE and not _has_date_hint(uv):
                continue
        else:
            if not st:
                continue
            if c == COL_CRM_DATE_PICKUP and not _has_date_hint(st):
                continue

        missing.append(c)

    missing_cols_per_row.append(missing)
    need_llm_mask.append(len(missing) > 0)

need_llm_df = df_final[pd.Series(need_llm_mask)].copy()

print('✓ LLM fill analysis (filtered):')
print(f'  Total rows: {len(df_final)}')
print(f'  Rows needing LLM: {len(need_llm_df)}')

# Quick stats: which columns are most missing
missing_counts = {c: 0 for c in LLM_TARGET_COLS}
for miss in missing_cols_per_row:
    for c in miss:
        missing_counts[c] += 1

missing_sorted = sorted(missing_counts.items(), key=lambda kv: kv[1], reverse=True)
print('\nTop missing columns:')
for c, n in missing_sorted[:10]:
    print(f'  {c}: {n}')

# Persist per-row missing list for later cells
df_final['_missing_cols'] = missing_cols_per_row

✓ LLM fill analysis (filtered):
  Total rows: 13803
  Rows needing LLM: 9979

Top missing columns:
  [Khách Hàng] Ý kiến KH: 9845
  [Khách Hàng] Nhận xét KH: 9842
  [Đối Thủ Cạnh Tranh] Các Hãng đối thủ cạnh tranh: 9446
  [AETT] Nhận xét tiếp thị: 9243
  [Hoạt Động CRM] Tiến độ: 9080
  [Đối Thủ Cạnh Tranh] Đối tượng: 8119
  [AETT] Đối tượng: 8119
  [Đối Thủ Cạnh Tranh] Lợi thế: 6219
  [Đối Thủ Cạnh Tranh] Nội dung làm việc: 6081
  [AETT] Nội dung làm việc: 6052


In [ ]:
# CELL 8: Prepare LLM Input JSON (only rows with missing/ambiguous cells)

LLM_INPUT_JSON = Path('llm_input.json')

items_for_llm: List[Dict[str, Any]] = []

for _, row in df_final.iterrows():
    missing_cols = row.get('_missing_cols', [])
    if not isinstance(missing_cols, list) or not missing_cols:
        continue

    locked_labels: Dict[str, Any] = {}
    for c in LLM_TARGET_COLS:
        v = row.get(c)
        if v is None:
            continue
        s = str(v).strip()
        if s and s.lower() != 'mơ hồ':
            locked_labels[c] = s

    items_for_llm.append({
        'row_idx': int(row['row_idx']),
        'R': row.get(COL_R, '') or '',
        'S': row.get(COL_S, '') or '',
        'T': row.get(COL_T, '') or '',
        'U': row.get(COL_U, '') or '',
        'V': row.get(COL_V, '') or '',
        'locked_labels': locked_labels,
        'missing_cols': missing_cols,
    })

LLM_INPUT_JSON.write_text(json.dumps(items_for_llm, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'✓ Saved LLM input: {len(items_for_llm)} rows → {LLM_INPUT_JSON.name}')
print('Next: run CELL 10 to call LLM and produce llm_fills.json')


✓ Saved LLM input: 9979 rows → llm_input.json
Next: run CELL 10 to call LLM and produce llm_fills.json


In [ ]:
# CELL X: Quick check available Gemini models (minimal output)
import os
import google.generativeai as genai

api_key = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY') or os.getenv('GOOGLE_GENERATIVEAI_API_KEY')
if not api_key:
    _k = globals().get('API_KEY')
    if isinstance(_k, str) and _k.strip():
        api_key = _k.strip()
        os.environ['GEMINI_API_KEY'] = api_key

genai.configure(api_key=api_key)
names = []
for m in genai.list_models():
    # keep only models that support generateContent
    if hasattr(m, 'supported_generation_methods') and 'generateContent' in (m.supported_generation_methods or []):
        names.append(m.name)
names = sorted(set(names))
print('models_generateContent=', len(names))
print('\n'.join(names[:20]))
if len(names) > 20:
    print('...')

models_generateContent= 34
models/deep-research-pro-preview-12-2025
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.5-computer-use-preview-10-2025
models/gemini-2.5-flash
models/gemini-2.5-flash-image
models/gemini-2.5-flash-image-preview
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro
models/gemini-2.5-pro-preview-tts
models/gemini-3-flash-preview
...


In [ ]:
# CELL 10: Run LLM to Fill All Blank/'mơ hồ' Cells (Checkpoint + Adaptive Batch)
import os
import re
import json
import time
import random
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import google.generativeai as genai

# ===== CONFIG =====
_k = globals().get('API_KEY')
if isinstance(_k, str) and _k.strip():
    API_KEY = _k.strip()
else:
    API_KEY = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY') or os.getenv('GOOGLE_GENERATIVEAI_API_KEY')
if not API_KEY:
    raise RuntimeError('Missing API_KEY (set API_KEY in kernel or env GEMINI_API_KEY)')

genai.configure(api_key=API_KEY)

MODEL_NAME = os.getenv('GEMINI_MODEL') or 'models/gemini-2.0-flash'

# Avoid 429: throttle between requests (seconds)
MIN_INTERVAL_S = float(os.getenv('GEMINI_MIN_INTERVAL_S') or '2.5')
JITTER_S = float(os.getenv('GEMINI_JITTER_S') or '0.5')
last_call = {'t': 0.0}

def _throttle() -> None:
    now = time.monotonic()
    elapsed = now - last_call['t']
    if elapsed < MIN_INTERVAL_S:
        time.sleep((MIN_INTERVAL_S - elapsed) + random.random() * JITTER_S)
    last_call['t'] = time.monotonic()

# Ensure LLM_TARGET_COLS exists even if CELL 7 was not run
if 'LLM_TARGET_COLS' not in globals():
    COL_CURRENT_STATUS = col_name('Hoạt Động CRM', 'Tình hình hiện tại')
    LLM_TARGET_COLS = [c for c in OUTPUT_COLUMNS if c != COL_CURRENT_STATUS]

PROMPT_BASE_PATH = Path('prompt_CRM_labeling_v4_full.txt')
LLM_INPUT_JSON = Path('llm_input.json')
OUT_LLM_JSON = Path('llm_fills.json')
CKPT_JSON = Path('llm_fills_checkpoint.json')

if not PROMPT_BASE_PATH.exists():
    raise RuntimeError(f'Missing prompt file: {PROMPT_BASE_PATH.name}')
if not LLM_INPUT_JSON.exists():
    raise RuntimeError(f'Missing input file: {LLM_INPUT_JSON.name} (run CELL 8 first)')

base_prompt = PROMPT_BASE_PATH.read_text(encoding='utf-8')
items: List[Dict[str, Any]] = json.loads(LLM_INPUT_JSON.read_text(encoding='utf-8'))
print(f'rows_for_llm={len(items)} model={MODEL_NAME} base_prompt_len={len(base_prompt)}')

# ===== Load keywords (for canonical labels + keyword hints) =====
if 'KW' in globals() and isinstance(globals().get('KW'), dict):
    KW_LOCAL = globals()['KW']
else:
    KW_LOCAL = json.loads(Path('keywords_fixed.json').read_text(encoding='utf-8'))

COL_PICKUP = col_name('Hoạt Động CRM', 'ngày lấy hàng')
COL_DATE_PLAN = col_name('Kế Hoạch', 'Ngày làm việc/ giao hàng:')
COL_BRANDS = col_name('Đối Thủ Cạnh Tranh', 'Các Hãng đối thủ cạnh tranh')

def _build_canonical_labels(kws: Dict[str, Any]) -> Tuple[Dict[str, List[str]], Dict[str, Dict[str, str]], List[str]]:
    """Return (canonical_list_by_col, canonical_lower_map_by_col, competitor_brands)."""
    canonical_by_col: Dict[str, List[str]] = {}
    lower_map_by_col: Dict[str, Dict[str, str]] = {}
    competitor_brands: List[str] = []

    for major, minors in (kws or {}).items():
        if not isinstance(minors, dict):
            continue
        for minor, subs in minors.items():
            # special nested competitor brands
            if major == 'Đối Thủ Cạnh Tranh' and minor == 'Cạnh Tranh' and isinstance(subs, dict):
                brand_list = subs.get('Các Hãng đối thủ cạnh tranh')
                if isinstance(brand_list, list):
                    for b in brand_list:
                        for term in split_terms(b):
                            if term:
                                competitor_brands.append(term.strip())
                continue

            # normal: minor -> {sub: [keywords]}
            if isinstance(subs, dict):
                col = col_name(major, minor)
                labels = [s for s in subs.keys() if isinstance(s, str) and s.strip()]
                if labels:
                    canonical_by_col[col] = labels
                    lm = {lbl.strip().lower(): lbl.strip() for lbl in labels}
                    lower_map_by_col[col] = lm

    # unique competitor brands
    seen = set()
    out_brands: List[str] = []
    for b in competitor_brands:
        k = b.lower()
        if k not in seen:
            out_brands.append(b)
            seen.add(k)

    return canonical_by_col, lower_map_by_col, out_brands

CANONICAL_BY_COL, CANONICAL_LOWER_MAP, COMPETITOR_BRANDS = _build_canonical_labels(KW_LOCAL)

def _sample_keywords(kws: Dict[str, Any], max_terms_per_sub: int = 3, max_subs_per_col: int = 6) -> str:
    lines: List[str] = []
    lines.append('=== CANONICAL LABELS (EXCERPT) ===')
    # list canonical labels for the 15 output columns (excluding copied status)
    for col in LLM_TARGET_COLS:
        if col in (COL_PICKUP, COL_DATE_PLAN, COL_BRANDS):
            continue
        labels = CANONICAL_BY_COL.get(col) or []
        if labels:
            lines.append(f'- {col}: ' + ', '.join(labels[:20]))

    lines.append('\n=== KEYWORD SAMPLES (EXCERPT) ===')

    # Provide a few sample keywords per (major, minor, sub)
    for major, minors in (kws or {}).items():
        if not isinstance(minors, dict):
            continue
        for minor, subs in minors.items():
            if major == 'Đối Thủ Cạnh Tranh' and minor == 'Cạnh Tranh':
                continue
            if not isinstance(subs, dict):
                continue
            col = col_name(major, minor)
            if col not in LLM_TARGET_COLS:
                continue
            shown = 0
            for sub, kw_list in subs.items():
                if shown >= max_subs_per_col:
                    break
                terms: List[str] = []
                if isinstance(kw_list, list):
                    for k in kw_list:
                        terms.extend(split_terms(k))
                elif isinstance(kw_list, str):
                    terms.extend(split_terms(kw_list))
                terms = [t for t in terms if t][:max_terms_per_sub]
                if terms:
                    lines.append(f'- {col} | {sub}: ' + '; '.join(terms))
                    shown += 1

    if COMPETITOR_BRANDS:
        lines.append('\n=== COMPETITOR BRANDS (SAMPLE) ===')
        lines.append('; '.join(COMPETITOR_BRANDS[:40]))

    return '\n'.join(lines)

keyword_hints = _sample_keywords(KW_LOCAL)
system_prompt = base_prompt + '\n\n' + keyword_hints
print(f'system_prompt_len={len(system_prompt)}')

# ===== Helpers: parse JSON from model =====
def _strip_code_fences(s: str) -> str:
    t = (s or '').strip()
    if t.startswith('```json'):
        t = t[len('```json'):].strip()
    elif t.startswith('```'):
        t = t[len('```'):].strip()
    if t.endswith('```'):
        t = t[:-3].strip()
    return t

def _extract_json_array(text: str) -> str:
    t = _strip_code_fences(text)
    if t.startswith('[') and t.endswith(']'):
        return t
    start = t.find('[')
    if start < 0:
        return t
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        ch = t[i]
        if in_str:
            if esc:
                esc = False
            elif ch == '\\':
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == '[':
                depth += 1
            elif ch == ']':
                depth -= 1
                if depth == 0:
                    return t[start:i+1]
    return t[start:]

def _parse_llm_json(text: str) -> List[Dict[str, Any]]:
    json_text = _extract_json_array(text)
    data = json.loads(json_text)
    if not isinstance(data, list):
        raise ValueError('Not a JSON array')
    for obj in data:
        if not isinstance(obj, dict) or 'row_idx' not in obj:
            raise ValueError('Each item must be an object with row_idx')
        if 'fills' not in obj or not isinstance(obj.get('fills'), dict):
            raise ValueError('Each item must contain fills object')
    return data

# ===== Date normalization =====
_DATE_DMY = re.compile(r'(?<!\d)(\d{1,2})[/-](\d{1,2})(?:[/-](\d{2,4}))?(?!\d)')
_DATE_YMD = re.compile(r'(?<!\d)(\d{4})-(\d{1,2})-(\d{1,2})(?!\d)')
_MONTH_YEAR = re.compile(r'(?<!\d)(\d{1,2})[/-](\d{4}|\d{2})(?!\d)')
_THANG = re.compile(r'(?:tháng|thang)\s*(\d{1,2})', re.IGNORECASE)

def _norm_ddmmyy(s: str) -> Optional[str]:
    if s is None:
        return None
    raw = str(s).strip()
    if not raw:
        return None

    t = raw.strip()

    m = _DATE_YMD.search(t)
    if m:
        yy = int(m.group(1)) % 100
        mm = int(m.group(2))
        dd = int(m.group(3))
        if 1 <= mm <= 12 and 1 <= dd <= 31:
            return f'{dd:02d}/{mm:02d}/{yy:02d}'

    m = _DATE_DMY.search(t)
    if m:
        dd = int(m.group(1))
        mm = int(m.group(2))
        yy_raw = m.group(3)
        if yy_raw is None:
            yy = 26
        else:
            yy = int(yy_raw) % 100
        if 1 <= mm <= 12 and 1 <= dd <= 31:
            return f'{dd:02d}/{mm:02d}/{yy:02d}'

    m = _MONTH_YEAR.search(t)
    if m:
        mm = int(m.group(1))
        yy = int(m.group(2)) % 100
        if 1 <= mm <= 12:
            return f'01/{mm:02d}/{yy:02d}'

    m = _THANG.search(t)
    if m:
        mm = int(m.group(1))
        if 1 <= mm <= 12:
            return f'01/{mm:02d}/26'

    return None

def _canonicalize_label(col: str, val: str) -> Optional[str]:
    if val is None:
        return None
    s = str(val).strip()
    if not s:
        return None

    # Date columns handled separately
    if col in (COL_PICKUP, COL_DATE_PLAN):
        return _norm_ddmmyy(s)

    # Competitor brands: allow any string (possibly multi-values separated by ;)
    if col == COL_BRANDS:
        return s

    # Other labels: must match canonical set
    lm = CANONICAL_LOWER_MAP.get(col) or {}
    return lm.get(s.lower())

# ===== Call model =====
model = genai.GenerativeModel(MODEL_NAME)

def call_llm_batch(batch: List[Dict[str, Any]], max_retry: int = 5) -> List[Dict[str, Any]]:
    payload = json.dumps(batch, ensure_ascii=False)
    user_input = 'INPUT_JSON_ARRAY:\n' + payload

    for attempt in range(1, max_retry + 1):
        try:
            _throttle()
            resp = model.generate_content(
                system_prompt + '\n\n' + user_input,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.0,
                    max_output_tokens=8192,
                ),
            )
            raw = getattr(resp, 'text', '') or ''
            return _parse_llm_json(raw)
        except Exception as e:
            msg = str(e)
            low = msg.lower()

            # 429 / rate limit handling
            if ('429' in low) or ('too many request' in low) or ('rate limit' in low) or ('perminute' in low):
                wait = min(120, 15 * attempt) + random.random() * 2
                print(f'⚠️ 429/rate-limit: wait={wait:.1f}s attempt={attempt} err={msg[:120]}')
                time.sleep(wait)
                continue

            # transient resource exhausted
            if ('resource_exhausted' in low) or ('temporarily' in low):
                wait = min(120, 10 * attempt) + random.random() * 2
                print(f'⚠️ transient: wait={wait:.1f}s attempt={attempt} err={msg[:120]}')
                time.sleep(wait)
                continue

            if attempt >= max_retry:
                raise

            wait = (2 ** attempt) + random.random()
            print(f'⚠️ retry={attempt} wait={wait:.1f}s err={msg[:120]}')
            time.sleep(wait)

    raise RuntimeError('LLM_FAILED')

# ===== Checkpoint =====
def load_checkpoint() -> Dict[str, Any]:
    if CKPT_JSON.exists():
        return json.loads(CKPT_JSON.read_text(encoding='utf-8'))
    return {'updated_at': None, 'results': {}}

def save_checkpoint(ckpt: Dict[str, Any]) -> None:
    ckpt['updated_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    CKPT_JSON.write_text(json.dumps(ckpt, ensure_ascii=False, indent=2), encoding='utf-8')

# Prepare lookup for allowed fill columns per row
missing_cols_map: Dict[str, set] = {}
for it in items:
    rid = str(int(it.get('row_idx')))
    miss = it.get('missing_cols')
    missing_cols_map[rid] = set(miss) if isinstance(miss, list) else set()

ckpt = load_checkpoint()
results: Dict[str, Any] = ckpt.get('results', {}) if isinstance(ckpt.get('results'), dict) else {}

done_ids = set(results.keys())
pending = [it for it in items if str(int(it['row_idx'])) not in done_ids]
print(f'checkpoint_done={len(done_ids)} pending={len(pending)}')

batch_size = int(os.getenv('GEMINI_BATCH_SIZE') or '20')
min_batch = 2
idx = 0

print('Bắt đầu gọi LLM...')
while idx < len(pending):
    cur_batch = pending[idx: idx + batch_size]
    try:
        print(f'{idx}/{len(pending)} bs={len(cur_batch)}', end=' ')
        out = call_llm_batch(cur_batch)

        out_by_id = {str(int(o.get('row_idx'))): o for o in out if isinstance(o, dict) and 'row_idx' in o}

        updated = 0
        for src in cur_batch:
            rid = str(int(src.get('row_idx')))
            allowed = missing_cols_map.get(rid, set())
            o = out_by_id.get(rid)
            if not o:
                continue
            fills = o.get('fills') or {}
            if not isinstance(fills, dict):
                fills = {}

            sanitized: Dict[str, Any] = {}
            for k, v in fills.items():
                if k not in allowed:
                    continue
                if v is None:
                    sanitized[k] = None
                    continue
                if isinstance(v, str) and not v.strip():
                    sanitized[k] = None
                    continue

                if isinstance(v, str):
                    sanitized[k] = _canonicalize_label(k, v)
                else:
                    sanitized[k] = None

            results[rid] = {'row_idx': int(rid), 'fills': sanitized}
            updated += 1

        ckpt['results'] = results
        save_checkpoint(ckpt)
        print(f'✓ updated={updated}')

        idx += batch_size

        # gentle pacing on top of throttle
        time.sleep(0.2)

    except Exception as e:
        err = str(e)
        print(f'✗ {err[:140]}')
        # If parse errors, shrink batch
        if ('not a json' in err.lower()) or ('json' in err.lower()) or ('length' in err.lower()):
            if batch_size > min_batch:
                batch_size = max(min_batch, batch_size // 2)
                print(f'  reduce_batch_size={batch_size}')
                time.sleep(2)
                continue
        raise

final_list = list(results.values())
final_list.sort(key=lambda x: int(x.get('row_idx', 0)))
OUT_LLM_JSON.write_text(json.dumps(final_list, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'\n✓ LLM done: {len(final_list)}/{len(items)} rows → {OUT_LLM_JSON.name}')
print(f'✓ Checkpoint: {CKPT_JSON.name}')


d:\Works\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\RD03590\AppData\Local\Temp\ipykernel_7372\2115650307.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


rows_for_llm=9979 model=models/gemini-2.0-flash base_prompt_len=4179
system_prompt_len=10886
checkpoint_done=0 pending=9979
Bắt đầu gọi LLM...
0/9979 bs=20 ✓ updated=20
20/9979 bs=20 ✓ updated=20
40/9979 bs=20 ✓ updated=20
60/9979 bs=20 ✓ updated=20
80/9979 bs=20 ✓ updated=20
100/9979 bs=20 ✓ updated=20
120/9979 bs=20 ✓ updated=20
140/9979 bs=20 ✓ updated=20
160/9979 bs=20 ✓ updated=20
180/9979 bs=20 ✓ updated=20
200/9979 bs=20 ✓ updated=20
220/9979 bs=20 ✓ updated=20
240/9979 bs=20 ✓ updated=20
260/9979 bs=20 ✓ updated=20
280/9979 bs=20 ✓ updated=20
300/9979 bs=20 ✓ updated=20
320/9979 bs=20 ✓ updated=20
340/9979 bs=20 ✓ updated=20
360/9979 bs=20 ✓ updated=20
380/9979 bs=20 ✓ updated=20
400/9979 bs=20 ✓ updated=20
420/9979 bs=20 ✓ updated=20
440/9979 bs=20 ✓ updated=20
460/9979 bs=20 ✓ updated=20
480/9979 bs=20 ✓ updated=20
500/9979 bs=20 ✓ updated=20
520/9979 bs=20 ✓ updated=20
540/9979 bs=20 ✓ updated=20
560/9979 bs=20 ✓ updated=20
580/9979 bs=20 ✓ updated=20
600/9979 bs=20 ✓ update

In [ ]:
# CELL 9: Merge LLM Fills + Export Final Excel

LLM_RESULTS_FILE = Path('llm_fills.json')
if not LLM_RESULTS_FILE.exists():
    print('ℹ️  llm_fills.json not found (run CELL 10 first)')
else:
    llm_results = json.loads(LLM_RESULTS_FILE.read_text(encoding='utf-8'))
    print(f'✓ Loaded {len(llm_results)} LLM rows')

    updated_total = 0
    updated_by_col: Dict[str, int] = {}

    def _is_blank_or_fuzzy(v: Any) -> bool:
        if v is None:
            return True
        s = str(v).strip()
        if not s:
            return True
        return s.lower() == 'mơ hồ'

    # Apply fills only when current value is blank or 'mơ hồ'
    for item in llm_results:
        if not isinstance(item, dict):
            continue
        row_idx = item.get('row_idx')
        fills = item.get('fills')
        if row_idx is None or not isinstance(fills, dict):
            continue

        rid = int(row_idx)
        mask = df_final['row_idx'] == rid
        if not mask.any():
            continue

        for col, val in fills.items():
            if col not in df_final.columns:
                continue
            if val is None:
                continue
            if _is_blank_or_fuzzy(df_final.loc[mask, col].iloc[0]):
                df_final.loc[mask, col] = val
                updated_total += 1
                updated_by_col[col] = updated_by_col.get(col, 0) + 1

    df_final.to_excel(PATH_OUTPUT_LLM, index=False)
    print(f'✓ Exported merged file: {PATH_OUTPUT_LLM.name}')
    print(f'  Updated cells: {updated_total}')

    top_cols = sorted(updated_by_col.items(), key=lambda kv: kv[1], reverse=True)[:12]
    if top_cols:
        print('  Top updated columns:')
        for c, n in top_cols:
            print(f'    {c}: {n}')


✓ Loaded 9979 LLM rows
✓ Exported merged file: CRM_classified_with_LLM.xlsx
  Updated cells: 17416
  Top updated columns:
    [AETT] Nội dung làm việc: 5077
    [Đối Thủ Cạnh Tranh] Nội dung làm việc: 4278
    [AETT] Nhận xét tiếp thị: 1590
    [AETT] Đối tượng: 1368
    [Đối Thủ Cạnh Tranh] Đối tượng: 1341
    [Kế Hoạch] Kế hoạch lần tới: 1208
    [Khách Hàng] Ý kiến KH: 823
    [Kế Hoạch] Đề xuất: 532
    [Hoạt Động CRM] Tiến độ: 528
    [Kế Hoạch] Ngày làm việc/ giao hàng:: 184
    [Khách Hàng] Nhận xét KH: 141
    [Đối Thủ Cạnh Tranh] Các Hãng đối thủ cạnh tranh: 131
